In [1]:
import os
import sys
print(sys.path) 

# sys.path.insert(0, "/mnt/nas/pritish/CMUVERA_IR_ref")
sys.path.insert(0, "/mnt/dgx/pritish/CMUVERA_IR_ref")
sys.path.insert(0, "/raid/infolab/pritish/CMUVERA_IR_ref")
print(sys.path)

['', '/mnt/home/pritish/.local/lib/python3.11/site-packages', '/mnt', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python311.zip', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/lib-dynload', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/site-packages', '/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/site-packages/setuptools/_vendor']
['/raid/infolab/pritish/CMUVERA_IR_ref', '/mnt/dgx/pritish/CMUVERA_IR_ref', '', '/mnt/home/pritish/.local/lib/python3.11/site-packages', '/mnt', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python311.zip', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/lib-dynload', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/site-packages', '/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/p

In [2]:
os.chdir("/mnt/dgx/data/pritish/CMUVERA_IR_ref")

In [3]:
import torch
import os
import numpy as np
from omegaconf import OmegaConf
import time
import logging
from src.utils import set_seed,set_seed_from_checkpoint, load, save
from tqdm import tqdm
import random
from numpy.random import default_rng
import submodlib
import pickle

from src.dataloader import get_dataloader
from src.embedder import ColBERTEmbedder

from src.utils import partial_chamfer_sim_batched_with_rerank
import torch.multiprocessing as mp

# 1) Make sure the env var is set *inside* Python too
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")

# 2) Turn on PyTorch’s deterministic‐only mode
torch.use_deterministic_algorithms(True)

from loguru import logger

/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/site-packages/beir/util.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [4]:
%load_ext autoreload
%autoreload 1

%aimport src.endtoend
%aimport make_plots

In [5]:
config = {
    "k": 15,
    "method": "v0",
    "bucket_size": 0,
    "dataset_name": "fever",
    "mv_type": 'colbertv2-plaid',
    "mode": "disk"
}

In [6]:
def make_config(config):
    sys.argv = ["", f"k={config['k']}", f"method={config['method']}",
                f"baseline.bucket_size={config['bucket_size']}",
                f"data.dataset_name={config['dataset_name']}",
                f"embedder.mv_type={config['mv_type']}",
                f"embedder.mode={config.get('mode', 'mem')}",
                "baseline.distributed_search=False"
    ]
    print(sys.argv)
    
    cli_conf = OmegaConf.from_cli()
    file_config = OmegaConf.load("configs/config.yaml")

    conf = OmegaConf.merge(file_config, cli_conf)

    return conf

In [7]:
conf = make_config(config)

['', 'k=15', 'method=v0', 'baseline.bucket_size=0', 'data.dataset_name=fever', 'embedder.mv_type=colbertv2-plaid', 'embedder.mode=disk', 'baseline.distributed_search=False']


In [8]:
def get_method(config):
    name = config.method
    if name == "v0":
        return src.endtoend.GreedyBaseline_v0(config)
    elif name=="sml":
        return src.endtoend.GreedyBaseline_submodlib(config)
    else:
        raise ValueError(f"Unknown method: {name}")

In [ ]:
retriever = get_method(conf)
retriever.run()

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5416568/5416568 [00:17<00:00, 308695.50it/s]


<function ColBERTEmbedder.embed_full_dataset.<locals>.<lambda> at 0x7f1080055da0>
./experiments/fever/BERT


/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()



#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: Ukrainian Soviet Socialist Republic was a founding participant of the UN., 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1,  5969,  3354,  6102,  3072,  2001,  1037,  4889, 13180,
         1997,  1996,  4895,  1012,   102,   103,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')



/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()


./experiments/fever/BERT/corpus/compressed_128/batch_0.0.pkl
./experiments/fever/BERT/corpus/compressed_128/batch_1.0.pkl
./experiments/fever/BERT/corpus/compressed_128/batch_2.0.pkl
./experiments/fever/BERT/corpus/compressed_128/batch_3.0.pkl
./experiments/fever/BERT/corpus/compressed_128/batch_4.0.pkl
./experiments/fever/BERT/corpus/compressed_128/batch_5.0.pkl
./experiments/fever/BERT/corpus/compressed_128/batch_6.0.pkl
./experiments/fever/BERT/corpus/compressed_128/batch_7.0.pkl
./experiments/fever/BERT/corpus/compressed_128/batch_8.0.pkl
./experiments/fever/BERT/corpus/compressed_128/batch_9.0.pkl
./experiments/fever/BERT/corpus/compressed_128/batch_10.0.pkl
./experiments/fever/BERT/corpus/compressed_128/batch_11.0.pkl
./experiments/fever/BERT/corpus/compressed_128/batch_12.0.pkl
./experiments/fever/BERT/corpus/compressed_128/batch_13.0.pkl
./experiments/fever/BERT/corpus/compressed_128/batch_14.0.pkl
./experiments/fever/BERT/corpus/compressed_128/batch_15.0.pkl
./experiments/feve

K:   0%|                                                                                                                                                       | 0/15 [00:00<?, ?it/s]/mnt/dgx/data/pritish/CMUVERA_IR_ref/src/embedder.py:288: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full c


KeyboardInterrupt



In [ ]:
# os.makedirs("plots/html",exist_ok=True)
# os.makedirs("plots/images",exist_ok=True)   
# # Load the data
# datasets = ["scifact"]
# b_sizes_aug = [10,25,50,100,200]
# k_values=[15]
# max_k = 15
# for data in datasets:
#     make_plots.plot_k_vs_metric(data,max_k,b_sizes_aug)
#     make_plots.plot_b_vs_metric(data,k_values,b_sizes_aug)

In [ ]:
with open("./pickles/results/greedy_base_0_128_k15_scifact_bf.pkl", "rb") as f:
    results = pickle.load(f)

In [ ]:
results[0]

[(2716, 11.888933181762695),
 (1858, 15.326838493347168),
 (2660, 17.4311466217041),
 (2022, 18.882389068603516),
 (1853, 19.841398239135742),
 (190, 20.44447898864746),
 (1321, 20.73671531677246),
 (916, 20.99953842163086),
 (4404, 21.165863037109375),
 (1421, 21.295223236083984),
 (1632, 21.38498306274414),
 (2967, 21.44413185119629),
 (1864, 21.495826721191406),
 (4658, 21.539615631103516),
 (2495, 21.570064544677734)]

In [ ]:
len(results)

300

In [ ]:
import copy
configg = copy.deepcopy(config)
configg["mode"] = "disk"
conff = make_config(configg)

['', 'k=15', 'method=v0', 'baseline.bucket_size=0', 'data.dataset_name=scifact', 'embedder.mv_type=colbertv2-plaid', 'embedder.mode=disk']


In [ ]:
retriever = get_method(conf)
retriever.run()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5183/5183 [00:00<00:00, 136759.88it/s]
/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
/mnt/nas/pritish/CMUVERA_IR_ref/src/embedder.py:327: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://


#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: 0-dimensional biomaterials show inductive properties., 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1,  1014,  1011,  8789, 16012,  8585, 14482,  2015,  2265,
        27427, 14194,  6024,  5144,  1012,   102,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')

./experiments/scifact/BERT/corpus/compressed_128/batch_0.0.pkl


/mnt/nas/pritish/CMUVERA_IR_ref/src/embedder.py:328: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cmask = torch.load(self.embedding_path("masks", i, j))["masks"]


0 1 loaded
0 2 loaded


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 300/300 [01:18<00:00,  3.82it/s]


[[(2716, 11.888933181762695),
  (1858, 15.326838493347168),
  (2660, 17.4311466217041),
  (2022, 18.882389068603516),
  (1853, 19.841398239135742),
  (190, 20.44447898864746),
  (1321, 20.73671531677246),
  (916, 20.99953842163086),
  (4404, 21.165863037109375),
  (1421, 21.295223236083984),
  (1632, 21.38498306274414),
  (2967, 21.44413185119629),
  (1864, 21.495826721191406),
  (4658, 21.539615631103516),
  (2495, 21.570064544677734)],
 [(816, 18.329320907592773),
  (2832, 20.94409942626953),
  (3044, 22.3422794342041),
  (2425, 23.02876091003418),
  (2621, 23.566795349121094),
  (2286, 23.767250061035156),
  (778, 23.924335479736328),
  (1876, 24.010135650634766),
  (2181, 24.094057083129883),
  (5123, 24.165117263793945),
  (3199, 24.22793960571289),
  (506, 24.287330627441406),
  (1956, 24.33557891845703),
  (1922, 24.36102294921875),
  (3444, 24.379596710205078)],
 [(2232, 15.830307960510254),
  (3462, 18.15846061706543),
  (3406, 20.152891159057617),
  (138, 20.897218704223633),

In [ ]:
with open("./pickles/results/greedy_base_0_128_k15_scifact_bf.pkl", "rb") as f:
    results_disk_mode = pickle.load(f)

In [ ]:
results_disk_mode[0]

[(2716, 11.888933181762695),
 (1858, 15.326838493347168),
 (2660, 17.4311466217041),
 (2022, 18.882389068603516),
 (1853, 19.841398239135742),
 (190, 20.44447898864746),
 (1321, 20.73671531677246),
 (916, 20.99953842163086),
 (4404, 21.165863037109375),
 (1421, 21.295223236083984),
 (1632, 21.38498306274414),
 (2967, 21.44413185119629),
 (1864, 21.495826721191406),
 (4658, 21.539615631103516),
 (2495, 21.570064544677734)]

In [ ]:
for i in range(len(results)):
    assert results[i] == results_disk_mode[i], f"Results mismatch at index {i}: {results[i]} != {results_disk_mode[i]}"

In [ ]:
retriever.embedder.cembs.shape

torch.Size([5183, 180, 128])

In [ ]:
retriever.embedder.qembs.permute(0, 2, 1).shape

torch.Size([300, 128, 32])

In [ ]:
prod = torch.matmul(retriever.embedder.cembs, retriever.embedder.qembs.permute(0, 2, 1))

RuntimeError: The size of tensor a (5183) must match the size of tensor b (300) at non-singleton dimension 0